# Compatibilidad Musical entre Dos Usuarios de Spotify

**Objetivo:** dado el perfil de audio de dos usuarios (sus canciones guardadas / top tracks), calcular:
1. Una puntuación de compatibilidad musical (0–100%)
2. En qué clusters escucha cada uno
3. Canciones 'puente' que ambos podrían disfrutar

Reutiliza el K-Means (k=2), PCA y StandardScaler del proyecto principal.
Los perfiles de usuario se simulan con el dataset existente (géneros preferidos) y pueden reemplazarse con datos reales de la API de Spotify.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "user_compatibility":
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "spotify_tracks_clean.csv")
df = df.reset_index(drop=True)

print(f"Dataset: {df.shape[0]:,} canciones | {df['track_genre'].nunique()} géneros")
print(f"Géneros disponibles: {sorted(df['track_genre'].unique())}")

## Features de Audio

Se usan exactamente las mismas features del notebook `06_clustering.ipynb` para mantener consistencia con el K-Means ya entrenado.

In [ ]:
CLUSTER_FEATURES = [
    "duration_min", "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

print("Features que definen el perfil de audio de un usuario:")
for f in CLUSTER_FEATURES:
    print(f"  - {f}")

## Simulación de Perfiles de Usuario

En producción, el perfil se construiría con los top tracks del usuario obtenidos de la API de Spotify:
```
GET /me/top/tracks  →  IDs de canciones
GET /audio-features/{id}  →  vector de features
perfil = media de los vectores de todas las canciones
```

Aquí simulamos dos usuarios con preferencias de género distintas y muestreamos canciones reales del dataset.

In [ ]:
# Usuario A: géneros urbanos / energéticos / bailables
CANDIDATE_A = ["hip-hop", "pop", "dance", "edm", "electronic", "disco",
               "funk", "r-n-b", "club", "party", "reggaeton", "dancehall"]

# Usuario B: géneros tranquilos / acústicos / instrumentales
CANDIDATE_B = ["acoustic", "classical", "ambient", "chill", "folk", "jazz",
               "new-age", "singer-songwriter", "piano", "study", "sleep"]

genres_a = [g for g in CANDIDATE_A if g in df["track_genre"].values]
genres_b = [g for g in CANDIDATE_B if g in df["track_genre"].values]

N = 50  # canciones por usuario (equivalente a un top-50 de Spotify)
df_a = df[df["track_genre"].isin(genres_a)].sample(n=N, random_state=RANDOM_STATE)
df_b = df[df["track_genre"].isin(genres_b)].sample(n=N, random_state=RANDOM_STATE + 1)

print(f"Usuario A — Géneros: {genres_a}")
print(f"\nMuestra de canciones Usuario A:")
print(df_a[["track_name", "artists", "track_genre"]].head(5).to_string(index=False))

print(f"\nUsuario B — Géneros: {genres_b}")
print(f"\nMuestra de canciones Usuario B:")
print(df_b[["track_name", "artists", "track_genre"]].head(5).to_string(index=False))

## Perfil de Audio por Usuario

El perfil es el **vector media** de todas las features de audio de sus canciones.
Representa el 'centro de gravedad' musical del usuario.

In [ ]:
profile_a = df_a[CLUSTER_FEATURES].mean()
profile_b = df_b[CLUSTER_FEATURES].mean()

profiles = pd.DataFrame({
    "Usuario A (Enérgico)": profile_a,
    "Usuario B (Acústico)": profile_b
}).round(4)

display(profiles)

In [ ]:
# Radar chart: comparación visual de perfiles
radar_features = ["danceability", "energy", "speechiness", "acousticness",
                  "instrumentalness", "liveness", "valence"]

n = len(radar_features)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for col, color, label in [
    ("Usuario A (Enérgico)", "#F58518", "Usuario A — Enérgico/Urbano"),
    ("Usuario B (Acústico)", "#4C78A8", "Usuario B — Acústico/Tranquilo")
]:
    vals = profiles.loc[radar_features, col].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, color=color, linewidth=2.5, label=label)
    ax.fill(angles, vals, color=color, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, size=12)
ax.set_title("Comparación de Perfiles de Audio", size=16, pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.12))
plt.tight_layout()
plt.show()

## Análisis de Clustering

Se aplica el mismo K-Means con k=2 del `notebook 06`, que identificó dos grandes clusters:
- **Cluster 0**: Enérgico / Bailable (alta energía, alta danceability, alta valence)
- **Cluster 1**: Acústico / Instrumental (alta acousticness, alta instrumentalness)

Esto permite ver en qué 'zona musical' vive cada usuario.

In [ ]:
scaler = StandardScaler()
X_all = scaler.fit_transform(df[CLUSTER_FEATURES])

kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_all)

CLUSTER_NAMES = {0: "Enérgico / Bailable", 1: "Acústico / Instrumental"}

dist_a = df.loc[df_a.index, "cluster"].value_counts(normalize=True).rename(CLUSTER_NAMES)
dist_b = df.loc[df_b.index, "cluster"].value_counts(normalize=True).rename(CLUSTER_NAMES)

print("Distribución de clusters — Usuario A:")
print(dist_a.round(3))
print("\nDistribución de clusters — Usuario B:")
print(dist_b.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#F58518", "#4C78A8"]

for ax, (dist, title) in zip(axes, [(dist_a, "Usuario A"), (dist_b, "Usuario B")]):
    ax.pie(dist.values, labels=dist.index, autopct="%1.1f%%",
           colors=colors, startangle=90, textprops={"fontsize": 11})
    ax.set_title(f"{title} — Distribución de Clusters", fontsize=13)

plt.suptitle("¿En qué cluster escucha cada usuario?", fontsize=14)
plt.tight_layout()
plt.show()

## Puntuación de Compatibilidad Musical

La **similitud coseno** mide el ángulo entre los dos vectores de perfil en el espacio de features.
- Valor 1.0 → perfiles idénticos
- Valor 0.0 → ortogonales (sin relación)
- Valor -1.0 → perfiles opuestos

Se normaliza al rango [0, 1] para producir una puntuación de compatibilidad interpretable.

In [ ]:
vec_a = scaler.transform(profile_a.values.reshape(1, -1))
vec_b = scaler.transform(profile_b.values.reshape(1, -1))

sim = cosine_similarity(vec_a, vec_b)[0][0]
compat = (sim + 1) / 2  # [-1, 1] → [0, 1]

if compat >= 0.80:
    interp = "Muy Alta — gustos musicales casi idénticos"
elif compat >= 0.65:
    interp = "Alta — compatibilidad significativa"
elif compat >= 0.50:
    interp = "Moderada — terreno común con diferencias notables"
elif compat >= 0.35:
    interp = "Baja — gustos diferentes con algo de overlap"
else:
    interp = "Muy Baja — preferencias musicales prácticamente opuestas"

print(f"Similitud coseno:             {sim:.4f}")
print(f"Puntuación de compatibilidad: {compat:.1%}")
print(f"Interpretación:               {interp}")

In [ ]:
# Bar chart: qué features los separan más
diff = (profile_a - profile_b).abs()
diff_norm = diff / diff.max()

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ["#E45756" if v > 0.5 else "#54A24B" for v in diff_norm.values]
ax.bar(diff_norm.index, diff_norm.values, color=bar_colors)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.7, label="Umbral de divergencia alta")
ax.set_title(f"Diferencias por Feature — Compatibilidad: {compat:.0%}", fontsize=13)
ax.set_ylabel("Diferencia Absoluta Normalizada")
ax.set_ylim(0, 1.1)
plt.xticks(rotation=30, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

## Recomendaciones Puente

El **punto medio** de ambos perfiles define un 'usuario hipotético' que representa el gusto compartido.
Las canciones más cercanas a ese punto en el espacio de features son las mejores candidatas para una **playlist conjunta**.

In [ ]:
midpoint_raw = ((profile_a + profile_b) / 2).values.reshape(1, -1)
midpoint_scaled = scaler.transform(midpoint_raw)

dists = np.linalg.norm(X_all - midpoint_scaled, axis=1)
df["dist_midpoint"] = dists

bridge = (
    df.nsmallest(10, "dist_midpoint")[["track_name", "artists", "track_genre", "popularity", "cluster"]]
    .copy()
)
bridge["cluster"] = bridge["cluster"].map(CLUSTER_NAMES)
bridge["popularity"] = bridge["popularity"].astype(int)

print("Top 10 canciones puente (más cercanas al punto medio de ambos perfiles):")
display(bridge.reset_index(drop=True))

In [ ]:
# Visualización PCA: posición de los usuarios y canciones puente
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_all)

rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(len(df), size=3000, replace=False)
bridge_idx = df.nsmallest(10, "dist_midpoint").index.values

fig, ax = plt.subplots(figsize=(11, 8))

# Fondo: muestra del dataset coloreada por cluster
ax.scatter(coords[sample_idx, 0], coords[sample_idx, 1],
           c=df.loc[sample_idx, "cluster"], cmap="Pastel1",
           alpha=0.3, s=8, label="_")

# Canciones de cada usuario
ax.scatter(coords[df_a.index, 0], coords[df_a.index, 1],
           c="#F58518", s=70, label="Usuario A", zorder=3,
           edgecolors="white", linewidths=0.5)
ax.scatter(coords[df_b.index, 0], coords[df_b.index, 1],
           c="#4C78A8", s=70, label="Usuario B", zorder=3,
           edgecolors="white", linewidths=0.5)

# Canciones puente
ax.scatter(coords[bridge_idx, 0], coords[bridge_idx, 1],
           c="#E45756", s=150, marker="*", label="Canciones puente", zorder=4)

# Punto medio
midpoint_2d = pca.transform(midpoint_raw)
ax.scatter(midpoint_2d[0, 0], midpoint_2d[0, 1],
           c="black", s=250, marker="X", label="Punto medio de gustos", zorder=5)

ax.set_title("Espacio PCA: Perfiles de Usuario y Canciones Puente", fontsize=14)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Integración con la API Real de Spotify

El siguiente bloque muestra cómo obtener perfiles de usuarios reales con la librería `spotipy`.
Está comentado porque requiere credenciales de una Spotify Developer App.

Una vez obtenidos `profile_a` y `profile_b` del API, el resto del análisis es idéntico a lo anterior.

In [ ]:
# Endpoints disponibles y su utilidad en este análisis
api_info = {
    "Endpoint": [
        "GET /me/top/tracks",
        "GET /me/tracks",
        "GET /me/player/recently-played",
        "GET /audio-features/{id}",
        "GET /me/top/artists",
        "GET /recommendations"
    ],
    "Datos obtenidos": [
        "Top 50 tracks (short / medium / long term)",
        "Canciones guardadas (liked songs)",
        "Últimas 50 canciones reproducidas",
        "danceability, energy, loudness, acousticness, etc.",
        "Artistas top + géneros asociados",
        "Tracks recomendados por seed de canciones/géneros"
    ],
    "Uso en el modelo": [
        "Construir perfil de audio del usuario",
        "Enriquecer el perfil con más canciones",
        "Capturar preferencias recientes",
        "Obtener el vector de features de cada canción",
        "Inferir géneros preferidos",
        "Usar canciones puente como seed para más recomendaciones"
    ]
}
display(pd.DataFrame(api_info))

In [ ]:
# ---------- CÓDIGO ILUSTRATIVO — requiere credenciales de Spotify ----------
#
# pip install spotipy
#
# import spotipy
# from spotipy.oauth2 import SpotifyOAuth
#
# sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
#     client_id="TU_CLIENT_ID",
#     client_secret="TU_CLIENT_SECRET",
#     redirect_uri="http://localhost:8888/callback",
#     scope="user-top-read user-library-read user-read-recently-played"
# ))
#
# def get_user_audio_profile(sp_client, limit=50, time_range="medium_term"):
#     """Retorna el vector promedio de features de audio del usuario."""
#     top = sp_client.current_user_top_tracks(limit=limit, time_range=time_range)
#     ids = [t["id"] for t in top["items"]]
#     features = sp_client.audio_features(ids)
#     feat_df = pd.DataFrame([f for f in features if f is not None])
#     audio_cols = ["danceability", "energy", "loudness", "speechiness",
#                   "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
#     # Nota: duration_min no viene del API — calcular con duration_ms / 60000
#     feat_df["duration_min"] = feat_df["duration_ms"] / 60000
#     return feat_df[CLUSTER_FEATURES].mean()
#
# # Reemplazar las líneas de simulación con:
# # profile_a = get_user_audio_profile(sp_user_a)
# # profile_b = get_user_audio_profile(sp_user_b)

print("Código ilustrativo — ver comentarios arriba para integración con API real.")

## Conclusiones

### Lo que produce este análisis
| Output | Descripción |
|--------|-------------|
| Puntuación de compatibilidad | Similitud coseno normalizada entre perfiles de audio |
| Distribución de clusters | % de canciones en cada zona musical (energética vs acústica) |
| Radar chart | Comparación visual feature por feature |
| Bar chart de divergencia | Qué dimensiones los separan más |
| Top 10 canciones puente | Candidatos para playlist conjunta |
| Visualización PCA | Posición espacial de ambos usuarios y canciones puente |

### Cómo extender
- **N usuarios**: calcular la matriz de compatibilidad cruzada y visualizarla como heatmap
- **Playlist dinámica**: generar N canciones puente y usarlas como seed en `GET /recommendations`
- **Compatibilidad en el tiempo**: recalcular con `recently_played` cada semana y ver si los gustos convergen
- **Clasificación de popularidad**: pasar las canciones puente por el modelo de red neuronal del notebook 05 para predecir si serán populares para ambos usuarios

### Modelos reutilizados del proyecto
| Notebook | Técnica | Aplicación aquí |
|----------|---------|------------------|
| `06_clustering.ipynb` | K-Means k=2 | Clasificar zona musical de cada usuario |
| `04_pca.ipynb` | PCA 2D | Visualizar perfiles en espacio reducido |
| `02_cleaning_feature_engineering.ipynb` | StandardScaler | Escalar antes de clustering y similitud |